## CS587 – Software Project Management (Phase 2)
## Cloud-Based Intelligent Resource Allocation System for Campus Facilities

**Domain:** Education

**Framework:** Ollama (local inference)

**Model:** Llama 3 (`gemma3:latest`)

**Approach:** Sequential Pipeline (Round Robin)

**Team:** Team 7

**Experiment Owner:** Nithishkannan Kuppal Thulasiraman - A20593857

#### Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests",
    "--quiet"
])
print("Libraries installed.")

Libraries installed.


#### Configuration & Setup

In [2]:
import os, json, time, re, requests
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Ollama configuration — runs locally, no API key needed
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:latest"

def check_ollama():
    """Verify Ollama is running and gemma3 model is available."""
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        if resp.status_code == 200:
            models = [m["name"] for m in resp.json().get("models", [])]
            if any("gemma3" in m for m in models):
                print(f"✅ Ollama running. gemma3 model found.")
                print(f"   Available models: {models}")
                return True
            else:
                print(f"⚠️  Ollama running but gemma3 not found.")
                print(f"   Available: {models}")
                print(f"   Run: ollama pull gemma3:12b")
                return False
        else:
            print("❌ Ollama not responding. Start it with: ollama serve")
            return False
    except Exception as e:
        print(f"❌ Cannot connect to Ollama: {e}")
        print("   Make sure Ollama is installed and running: ollama serve")
        return False

ollama_ready = check_ollama()

def call_ollama(system_prompt: str, user_message: str, max_tokens: int = 1500) -> str:
    """Call Ollama API with system + user message, return response text."""
    full_prompt = f"<start_of_turn>system\n{system_prompt}<end_of_turn>\n<start_of_turn>user\n{user_message}<end_of_turn>\n<start_of_turn>model\n"
    payload = {
        "model": MODEL,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": 0.4,
            "num_predict": max_tokens,
            "top_p": 0.9,
            "stop": ["<end_of_turn>", "<start_of_turn>"]
        }
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=600)
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except requests.exceptions.Timeout:
        return "[ERROR: Ollama request timed out after 600 seconds]"
    except Exception as e:
        return f"[ERROR: {e}]"

print(f"Model: {MODEL}")
print(f"Ollama URL: {OLLAMA_URL}")
print("Configuration complete.")

✅ Ollama running. gemma3 model found.
   Available models: ['gemma3:latest', 'llama3.1:latest', 'llama3:latest', 'llama3.2:3b-instruct-fp16', 'llama3.2:1b', 'llama3.2:3b']
Model: gemma3:latest
Ollama URL: http://localhost:11434/api/generate
Configuration complete.


#### Project Domain & Scrum Configuration

In [3]:
DOMAIN = """
PROJECT: Cloud-Based Intelligent Resource Allocation System for Campus Facilities
DOMAIN: Education
DESCRIPTION: Students and faculty book rooms, labs, and equipment online.
Administrators view real-time occupancy dashboards. AI/ML predicts demand and
auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook,
and SSO. Sends booking/conflict notifications. Generates utilization reports.
Supports 5,000+ concurrent users and maintains 99.9% uptime.
STAKEHOLDERS: Students, Faculty, Facility Admins, IT Department, University Management
FACILITIES: Classrooms, Computer Labs, Research Labs, Conference Rooms, Study Rooms
"""

SCRUM_CONFIG = """
SCRUM FRAMEWORK:
- Sprint Duration: 2 weeks (10 business days)
- Total Sprints: 4 sprints
- Sprint Velocity: 40 story points per sprint
- Story Point Scale: Fibonacci (3, 5, 8)
- Definition of Done: Code reviewed, unit tested, integrated, demo-ready
- Ceremonies: Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective
- Team: Scrum Master, Product Owner, Developers, QA, DevOps, UI/UX Designer
"""
print("Domain and Scrum config loaded.")

Domain and Scrum config loaded.


#### Define Scrum AI Agents 

In [4]:
AGENT_SYSTEM_PROMPTS = {

"Customer_Proxy": """You are the Customer Proxy representing the university client.
Start your response with: '## CUSTOMER PROXY OUTPUT'

Provide a clear product vision statement including:
- Business problem and project goals
- Key success metrics (uptime, concurrent users, integrations)
- High-level scope from the customer perspective

Keep your response focused and concise.""",

"Product_Owner": """You are the Product Owner for a Scrum-based campus facility system.
IMPORTANT: Keep total response under 800 words. Show ONLY the final sprint table.
Start your response with: '## PRODUCT OWNER OUTPUT'
"CRITICAL: EVERY story must appear in exactly one sprint. Sprint totals must be 36-40 SP each. Verify: count of stories in sprints = count in backlog.\n"

Your deliverables:
1. Product Backlog with AT LEAST 24 user stories
   - Each story: ID, user story, acceptance criteria, priority, story points
   - Group into Epics: Authentication, Booking, AI/ML, Notifications, Reporting, Admin
   - Use ONLY Fibonacci story points: 3, 5, or 8 (no 13, no 21)

2. Sprint Plan for EXACTLY 4 sprints
   - Velocity = 40 story points per sprint
   - Each sprint MUST total between 36 and 40 story points
   - Every story assigned to exactly ONE sprint — NO story left unassigned
   - Each sprint MUST have AT LEAST 6 stories
   - After assigning stories, ADD UP the SP values for each sprint to verify
   - Example correct sprint: AUTH-01(8)+BOOK-01(5)+AI-01(8)+NOTIF-01(3)+REP-01(8)+ADMIN-01(5) = 37 SP ✓
   - Example wrong sprint: AUTH-01(5)+BOOK-01(8)+NOTIF-01(3) = 16 SP ✗ TOO LOW — add more stories
   - If sprint total is below 36 SP — move stories FROM other sprints INTO this sprint
   - If sprint total is above 40 SP — move one story TO the next sprint
   - Show ONLY the final verified sprint table
   - Final verification: Sprint1 + Sprint2 + Sprint3 + Sprint4 = Grand Total

3. Definition of Done

4. Effort Calculation:
   - Total Work = Sprint1 total + Sprint2 total + Sprint3 total + Sprint4 total
   - Do NOT recount from backlog separately
   - Total Work = ___ story points
   - Velocity = 40 story points/sprint
   - Effort = 4 sprints

End with a section titled 'Effort Estimate'.""",

"Scrum_Master": """You are the Scrum Master for a Scrum-based campus facility project.
Start your response with: '## SCRUM MASTER OUTPUT'

Your deliverables:
1. Scrum Ceremonies definition:
   - Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective
   - Each sprint: 1 Planning + 10 Daily Scrums + 1 Review + 1 Retrospective = 13 ceremonies
   - 4 sprints x 13 ceremonies = 52 total ceremonies

2. Sprint Goals for ALL 4 sprints
   - Copy EXACTLY the Story IDs from the Product Owner sprint table above
   - Do NOT add, remove, or move any stories
   - Format: Sprint N Goal: "...", Stories: AUTH-01, BOOK-02..., Sprint Total: XX SP
   - Sprint totals MUST match Product Owner sprint totals exactly

3. Velocity tracking and burndown plan

4. Effort Calculation:
   - Total Work = 52 ceremonies
   - Productivity = 3 ceremonies/day
   - Effort = 52 / 3 = 17.33 days
   - NEVER divide by 13. Productivity is always 3 ceremonies/day, not 13.

End with a section titled 'Effort Estimate'.""",

"Business_Analyst": """You are the Business Analyst for a Scrum-based campus facility project.
IMPORTANT: Keep total response under 600 words. Use one-line Given-When-Then per story.
Start your response with: '## BUSINESS ANALYST OUTPUT'

Your deliverables:
1. Detailed user stories with Given-When-Then acceptance criteria for ALL stories
2. Requirement Traceability Matrix (Story ID → Requirement → Sprint → Priority → Status)
3. Dependencies, assumptions, constraints, and compliance concerns
4. Business rules and success criteria

Effort Calculation:
- Total Work = total number of user stories documented
- Productivity = 5 requirements/day
- Effort = Total Work / 5

End with a section titled 'Effort Estimate'.""",

"UI_UX_Designer": """You are the UI/UX Designer for a Scrum-based campus facility project.
Start your response with: '## UI/UX DESIGNER OUTPUT'

Your deliverables:
1. User flows for Student, Instructor, and Admin roles
2. Wireframe descriptions for AT LEAST 12 screens
   - Label each screen: S1, S2, S3... up to S12 or more
   - Include: Login, Dashboard, Room Search, Booking, Confirmation, My Bookings,
     Admin Dashboard, Occupancy View, Reports, Notifications, Calendar Integration, Settings
3. Design system: typography, color palette, component library
4. WCAG 2.1 AA accessibility compliance plan
5. Sprint-wise design tasks for all 4 sprints

Effort Calculation:
- Total Work = total number of screens designed
- Productivity = 3 screens/day
- Effort = Total Work / 3

End with a section titled 'Effort Estimate'.""",

"Developer": """You are the Developer for a Scrum-based campus facility project.
Start your response with: '## DEVELOPER OUTPUT'

Your deliverables:
1. Development tasks mapped to Product Backlog Story IDs (DEV-01, DEV-02...)
2. System architecture: Frontend (React), Backend (Node/Express), DB (MongoDB), AI/ML (Python)
3. API design: endpoints for User, Booking, Admin, AI, Integration, Notification
4. Sprint-by-sprint implementation plan
   - Copy EXACTLY the Story IDs and sprint groupings from the Product Owner sprint table
   - Do NOT regroup, reassign, or move any stories to different sprints
   - Each sprint SP total in your table MUST match the Product Owner exactly
   - Show Story IDs and SP per sprint to verify totals match
5. Quality gates: code review, linting, unit tests, integration tests

Effort Calculation:
- Total Work = same story points as Product Backlog grand total
- Individual Velocity = 8 story points/developer/sprint
- Team Velocity = 40 story points/sprint (5 developers x 8)
- Effort = Total Work / 40 → round to 4 sprints

End with a section titled 'Effort Estimate'.""",

"QA_Engineer": """You are the QA Engineer for a Scrum-based campus facility project.
IMPORTANT: Keep total response under 600 words. Use compact one-line table format.
Start your response with: '## QA ENGINEER OUTPUT'
"CRITICAL: You MUST output a numbered table of 20+ test cases with columns: TC ID, Description, Story ID, Expected Result, Priority. No prose. No summaries. Just the table.\n"
Your deliverables:
1. Test strategy covering functional, non-functional, performance, security, AI quality
2. AT LEAST 20 Functional Test Cases (TC-F-01, TC-F-02...)
   - Each: TC ID, Description, Related Story ID, Steps, Expected Result, Priority
3. AT LEAST 7 Non-Functional Test Cases (TC-NF-01, TC-NF-02...)
   - Cover: performance (5000+ users), uptime (99.9%), security, accessibility (WCAG 2.1 AA)
4. Full Traceability Matrix: Story ID → Test Case IDs
5. Regression plan and release quality criteria
6. Sprint-aligned testing schedule for all 4 sprints

Effort Calculation:
- Total Work = total number of test cases
- Productivity = 3 test cases/day
- Effort = Total Work / 3

End with a section titled 'Effort Estimate'.""",

"DevOps_Engineer": """You are the DevOps Engineer for a Scrum-based campus facility project.
Start your response with: '## DEVOPS ENGINEER OUTPUT'
"CRITICAL: Output ONLY a table of 16+ DevOps tasks. Format: | DO-01 | Task | Sprint | Stories |. Do NOT write a status report. Do NOT summarize other agents.\n"

Your deliverables:
1. CI/CD Pipeline: GitHub Actions → Docker → Kubernetes (Dev/Staging/Production)
2. Sprint-aligned DevOps tasks table with numbered IDs:
   | Task ID | DevOps Task | Sprint | Stories Supported |
   Include at least 16 tasks (DO-01 through DO-16 or more)
3. Monitoring: Prometheus + Grafana, ELK Stack logging
4. Security: image scanning (Trivy), secrets management (Vault)
5. Rollback strategy and operational readiness checklist

Effort Calculation:
- Total Work = total number of DO-XX tasks
- Productivity = 8 tasks/day
- Effort = Total Work / 8

End with a section titled 'Effort Estimate'."""
}

print("Scrum AI Agent prompts defined.")
for name in AGENT_SYSTEM_PROMPTS:
    print(f"  - {name}")

Scrum AI Agent prompts defined.
  - Customer_Proxy
  - Product_Owner
  - Scrum_Master
  - Business_Analyst
  - UI_UX_Designer
  - Developer
  - QA_Engineer
  - DevOps_Engineer


#### Customer Problem Statement

In [5]:
customer_message = (
    "We need a Scrum-based project plan for our Cloud-Based Intelligent "
    "Resource Allocation System for Campus Facilities.\n\n"
    "Each agent must respond ONLY with their own role-specific Scrum artifacts.\n\n"
    "Product Owner: Create the Product Backlog with AT LEAST 24 user stories. "
    "Plan EXACTLY 4 sprints at 40 SP velocity. Each sprint MUST total 36-40 SP. "
    "Use only Fibonacci points: 3, 5, or 8. Show Story IDs in sprint table. "
    "Each sprint needs AT LEAST 6 stories. "
    "Verify each sprint total by adding SP values before finalizing. "
    "Show ONLY the final verified sprint table.\n"
    "Scrum Master: Define ceremonies and sprint goals for all 4 sprints. "
    "Use exactly 10 Daily Scrums per sprint = 52 total ceremonies. "
    "List Story IDs in each sprint goal.\n"
    "Business Analyst: Write Given-When-Then acceptance criteria and traceability matrix.\n"
    "UI/UX Designer: Define wireframes for 12+ screens with IDs (S1, S2...) and design system.\n"
    "Developer: Define dev tasks (DEV-01...), architecture, APIs. "
    "Follow the Product Owner sprint assignments exactly.\n"
    "QA Engineer: Create 20+ functional (TC-F-01...) and 7+ non-functional test cases "
    "with full traceability matrix.\n"
    "DevOps Engineer: Design CI/CD pipeline with task IDs (DO-01, DO-02...) "
    "in a sprint-aligned table, at least 16 tasks.\n\n"
    "Project Scope: Students and faculty book rooms, labs, and equipment online. "
    "Administrators view real-time occupancy dashboards. AI/ML predicts demand and "
    "auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook, "
    "and SSO. Sends booking and conflict notifications. Generates utilization reports. "
    "Supports 5,000+ concurrent users and maintains 99.9% uptime."
    "\nAll agents MUST use ONLY these story ID prefixes: AUTH, BOOK, AI, NOTIF, REP, ADMIN. "
    "Number them AUTH-01, AUTH-02, BOOK-01 etc. Never invent other prefixes."
)
print("Customer statement loaded.")

Customer statement loaded.


#### Run Experiment

In [6]:
agent_names = [
    "Product_Owner", "Scrum_Master", "Business_Analyst",
    "UI_UX_Designer", "Developer", "QA_Engineer", "DevOps_Engineer"
]

# Build conversation history — each agent sees the customer message + all prior responses
conversation_history = customer_message
all_results = []
po_output = ""

print("Starting Ollama (Gemma3) Sequential Pipeline experiment...")
print("=" * 60)

start_time = time.time()

for agent_name in agent_names:
    print(f"\n>>> Running: {agent_name}")
    agent_start = time.time()

    # Reset context for Developer, QA, DevOps to avoid overflow
    if agent_name == "Developer":
        conversation_history = customer_message + "\n\n---\n\nProduct_Owner Output:\n" + po_output

    system_prompt = AGENT_SYSTEM_PROMPTS[agent_name]

    if agent_name in ["Product_Owner", "Developer", "QA_Engineer", "DevOps_Engineer"]:
        agent_response = call_ollama(system_prompt, conversation_history, max_tokens=2000)
    else:
        agent_response = call_ollama(system_prompt, conversation_history, max_tokens=1000)

    agent_elapsed = time.time() - agent_start

    all_results.append({
        "agent": agent_name,
        "response": agent_response,
        "time": agent_elapsed,
        "model": MODEL,
    })

    # Save PO output for context reset
    if agent_name == "Product_Owner":
        po_output = agent_response

    # Append this agent's output to conversation history for next agent
    conversation_history += f"\n\n----\n\n{agent_name} Output:\n{agent_response}"

    print(f"    Done in {agent_elapsed:.1f}s | {len(agent_response):,} chars")
    print(f"    Preview: {agent_response[:100].strip()}...")

total_elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"  Ollama (Gemma3) Sequential Pipeline completed in {total_elapsed:.1f}s")
print("=" * 60)

Starting Ollama (Gemma3) Sequential Pipeline experiment...

>>> Running: Product_Owner
    Done in 117.2s | 4,703 chars
    Preview: ## PRODUCT OWNER OUTPUT

**Product Backlog:**

| Story ID | User Story...

>>> Running: Scrum_Master
    Done in 91.2s | 1,781 chars
    Preview: ## SCRUM MASTER OUTPUT

**1. Scrum Ceremonies Definition:**

*   **Sprint Planning:** 1 per sprint –...

>>> Running: Business_Analyst
    Done in 136.4s | 5,887 chars
    Preview: ## BUSINESS ANALYST OUTPUT

**1. User Stories with Given-When-Then Acceptance Criteria:**

| Story I...

>>> Running: UI_UX_Designer
    Done in 186.6s | 3,668 chars
    Preview: ## UI/UX DESIGNER OUTPUT

**1. User Flows:**

*   **Student Booking Flow:**
    1.  Login (S1)
    2...

>>> Running: Developer
    Done in 236.6s | 6,738 chars
    Preview: ## DEVELOPER OUTPUT

**1. Development Tasks Mapped to Product Backlog Story IDs:**

| DEV Task ID |...

>>> Running: QA_Engineer
    Done in 426.9s | 4,459 chars
    Preview: ## QA ENGINE

#### Save Results

In [7]:
import os, json

output_dir = "Experiment_Results_3"
os.makedirs(output_dir, exist_ok=True)

total_chars = sum(len(r["response"]) for r in all_results)

section_names = {
    "Product_Owner":    "SECTION 1 – PRODUCT OWNER (PRODUCT BACKLOG)",
    "Scrum_Master":     "SECTION 2 – SCRUM MASTER (CEREMONIES & SPRINT GOALS)",
    "Business_Analyst": "SECTION 3 – BUSINESS ANALYST (USER STORIES & TRACEABILITY)",
    "UI_UX_Designer":   "SECTION 4 – UI/UX DESIGNER (WIREFRAMES & DESIGN SYSTEM)",
    "Developer":        "SECTION 5 – DEVELOPER (SPRINT TASKS & ARCHITECTURE)",
    "QA_Engineer":      "SECTION 6 – QA ENGINEER (TEST PLAN)",
    "DevOps_Engineer":  "SECTION 7 – DEVOPS ENGINEER (CI/CD & INFRASTRUCTURE)",
}

# Save individual agent outputs
for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']} | **Time:** {result['time']:.1f}s\n\n")
        f.write("---\n\n")
        f.write(result["response"])
    print(f"Saved: {filepath}")

# Save combined report
combined_path = os.path.join(output_dir, "Phase2_Scrum_Complete_Report.md")
with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Phase 2 – Scrum Project Plan\n")
    f.write("# Cloud-Based Intelligent Resource Allocation System for Campus Facilities\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**LLM Model:** {MODEL}\n")
    f.write("**Framework:** Ollama (Gemma3) – Sequential Pipeline\n")
    f.write("**Methodology:** Scrum (Agile Iterative Development)\n\n---\n\n")
    f.write("## SECTION 0 – CUSTOMER PROBLEM STATEMENT\n\n")
    f.write(customer_message + "\n\n---\n\n")
    for result in all_results:
        section = section_names.get(result["agent"], result["agent"])
        f.write(f"## {section}\n\n")
        f.write(f"**Model:** {result['model']} | **Time:** {result['time']:.1f}s\n\n")
        f.write(result["response"] + "\n\n---\n\n")

# Save metadata
metadata = {
    "experiment": {
        "title": "Phase 2 – Scrum Project Planning",
        "project": "Cloud-Based Intelligent Resource Allocation System",
        "model": MODEL,
        "framework": "Ollama (Gemma3)  – Sequential Pipeline",
        "interaction": "Sequential Pipeline",
        "date": datetime.now().isoformat(),
        "total_time_seconds": round(total_elapsed, 2),
        "total_chars": total_chars,
        "cost": "$0.00 (local inference)",
    },
    "agents": [{"name": r["agent"], "time": round(r["time"], 2), "model": r["model"]} for r in all_results],
}
metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"\nCombined report saved: {combined_path}")
print(f"Metadata saved: {metadata_path}")
print(f"Total characters: {total_chars:,}")

Saved: Experiment_Results_3\Product_Owner_output.md
Saved: Experiment_Results_3\Scrum_Master_output.md
Saved: Experiment_Results_3\Business_Analyst_output.md
Saved: Experiment_Results_3\UI_UX_Designer_output.md
Saved: Experiment_Results_3\Developer_output.md
Saved: Experiment_Results_3\QA_Engineer_output.md
Saved: Experiment_Results_3\DevOps_Engineer_output.md

Combined report saved: Experiment_Results_3\Phase2_Scrum_Complete_Report.md
Metadata saved: Experiment_Results_3\experiment_metadata.json
Total characters: 35,026


#### Work Effort, Duration & Effort Analysis

In [8]:
HOURS_PER_PT = 4
WD = 8
dur_min = total_elapsed / 60
avg_per_agent = total_elapsed / len(all_results) if all_results else 0
NUM_SPRINTS = 4
SPRINT_WEEKS = 2
proj_weeks = NUM_SPRINTS * SPRINT_WEEKS
proj_budget_hrs = proj_weeks * 5 * WD

print('=' * 62)
print('  WORK EFFORT, DURATION & EFFORT ANALYSIS')
print('=' * 62)
print(f'\n📅 DURATION')
print(f'  Sprints Planned         : {NUM_SPRINTS} x {SPRINT_WEEKS}-week sprints')
print(f'  Total Project Duration  : {proj_weeks} weeks ({proj_weeks*5} working days / {proj_weeks*7} calendar days)')
print(f'  Capacity Budget         : {proj_budget_hrs} work-hours')
print(f'  AI Experiment (wall-clock): {total_elapsed:.1f}s  ({dur_min:.1f} min)')
print(f'  Avg Time per Agent      : {avg_per_agent:.1f}s')

# Extract story points from PO response
po = next((r['response'] for r in all_results if r['agent'] == 'Product_Owner'), '')

# Try multiple regex patterns to find total SP
total_sp_match = re.search(r'Total Work\s*[=:]\s*(\d+)\s*story points', po, re.IGNORECASE)
if not total_sp_match:
    total_sp_match = re.search(r'\*{0,2}(\d+)\*{0,2}\s*story points', po, re.IGNORECASE)
if not total_sp_match:
    # Try to sum sprint totals
    sprint_totals = re.findall(r'Sprint\s*\d+.*?[:|]\s*(\d+)\s*SP', po, re.IGNORECASE)
    total_sp = sum(int(x) for x in sprint_totals) if sprint_totals else 0
else:
    total_sp = int(total_sp_match.group(1))

avg_vel = total_sp / NUM_SPRINTS if total_sp else 0

print(f'\n🎯 WORK EFFORT  (Story Points)')
print(f'  Total Story Points      : {total_sp} pts')
print(f'  Team Velocity           : 40 story points/sprint')
print(f'  Sprints Needed          : {NUM_SPRINTS} sprints')
print(f'  Avg Velocity            : {avg_vel:.1f} pts/sprint')

print(f'\n⏱  EFFORT ESTIMATION  (@ {HOURS_PER_PT}h per story point)')
th = total_sp * HOURS_PER_PT
print(f'  Total Estimated Effort  : {th} hrs  ({th/WD:.1f} days)  ({th/WD/5:.1f} weeks)')
print(f'  Capacity Utilisation    : {th/proj_budget_hrs*100:.1f}%   ({th}/{proj_budget_hrs} hrs)')

print(f'\n🤖 AGENT TIME EFFORT')
print(f"  {'Agent':<22} {'Time(s)':>8} {'%':>6} {'Chars':>8}")
print(f"  {'-'*50}")
for r in all_results:
    pct = r['time'] / total_elapsed * 100 if total_elapsed else 0
    print(f"  {r['agent']:<22} {r['time']:>8.1f}  {pct:>5.1f}%  {len(r['response']):>8,}")
print(f"  {'-'*50}")
print(f"  {'TOTAL':<22} {total_elapsed:>8.1f}  100.0%  {total_chars:>8,}")
print(f"  Cost: $0.00 USD (Ollama local inference — FREE)")
print('=' * 62)

  WORK EFFORT, DURATION & EFFORT ANALYSIS

📅 DURATION
  Sprints Planned         : 4 x 2-week sprints
  Total Project Duration  : 8 weeks (40 working days / 56 calendar days)
  Capacity Budget         : 320 work-hours
  AI Experiment (wall-clock): 1468.0s  (24.5 min)
  Avg Time per Agent      : 209.7s

🎯 WORK EFFORT  (Story Points)
  Total Story Points      : 171 pts
  Team Velocity           : 40 story points/sprint
  Sprints Needed          : 4 sprints
  Avg Velocity            : 42.8 pts/sprint

⏱  EFFORT ESTIMATION  (@ 4h per story point)
  Total Estimated Effort  : 684 hrs  (85.5 days)  (17.1 weeks)
  Capacity Utilisation    : 213.8%   (684/320 hrs)

🤖 AGENT TIME EFFORT
  Agent                   Time(s)      %    Chars
  --------------------------------------------------
  Product_Owner             117.2    8.0%     4,703
  Scrum_Master               91.2    6.2%     1,781
  Business_Analyst          136.4    9.3%     5,887
  UI_UX_Designer            186.6   12.7%     3,668
  Deve

#### Experiment Summary

In [9]:
print('=' * 62)
print('  SCRUM EXPERIMENT 3 COMPLETE')
print('=' * 62)
print()
print(f'  Model         : {MODEL}')
print(f'  Framework     : Ollama – Gemma3 (local inference)')
print(f'  Method        : Scrum (Agile) – 4 Sprints')
print(f'  Interaction   : Sequential Pipeline')
print(f'  Total Agents  : {len(all_results)}')
print(f'  Total Time    : {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)')
print(f'  Cost          : $0.00 (local, FREE)')
print()
print(f'  -- Work Effort --')
print(f'  Total Story Points : {total_sp} pts')
print(f'  Team Velocity      : 40 story points/sprint')
print(f'  Sprints Needed     : {NUM_SPRINTS} sprints')
print(f'  Total Effort       : {total_sp * 4} hrs  ({total_sp * 4 / 8:.1f} days)')
print(f'  Avg Velocity       : {total_sp / NUM_SPRINTS:.1f} pts/sprint')
print()
print(f'  -- Duration --')
print(f'  Scrum Plan   : {NUM_SPRINTS} sprints x {SPRINT_WEEKS} weeks = {proj_weeks} weeks')
print(f'  Working Days : {proj_weeks*5} working days / {proj_weeks*7} calendar days')
print(f'  AI Experiment: {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)')
print(f'  Avg/Agent    : {total_elapsed/len(all_results):.1f}s')
print()
print(f'  -- Agent Breakdown --')
for r in all_results:
    print(f"    {r['agent']:<22s} | Time: {r['time']:>6.1f}s | Chars: {len(r['response']):>6,}")
print()
print(f'  Total Characters Used: {total_chars:,}')
print(f'  Output Dir: {output_dir}/')
print('=' * 62)

  SCRUM EXPERIMENT 3 COMPLETE

  Model         : gemma3:latest
  Framework     : Ollama – Gemma3 (local inference)
  Method        : Scrum (Agile) – 4 Sprints
  Interaction   : Sequential Pipeline
  Total Agents  : 7
  Total Time    : 1468.0s  (24.5 min)
  Cost          : $0.00 (local, FREE)

  -- Work Effort --
  Total Story Points : 171 pts
  Team Velocity      : 40 story points/sprint
  Sprints Needed     : 4 sprints
  Total Effort       : 684 hrs  (85.5 days)
  Avg Velocity       : 42.8 pts/sprint

  -- Duration --
  Scrum Plan   : 4 sprints x 2 weeks = 8 weeks
  Working Days : 40 working days / 56 calendar days
  AI Experiment: 1468.0s  (24.5 min)
  Avg/Agent    : 209.7s

  -- Agent Breakdown --
    Product_Owner          | Time:  117.2s | Chars:  4,703
    Scrum_Master           | Time:   91.2s | Chars:  1,781
    Business_Analyst       | Time:  136.4s | Chars:  5,887
    UI_UX_Designer         | Time:  186.6s | Chars:  3,668
    Developer              | Time:  236.6s | Chars:  6